# 07 — Figure 7: peak X-ray flux vs. flare recurrence time

Reproduces **Figure 7** of the paper — the central result that flare
**recurrence time is decoupled from individual flare energetics**
(Spearman ρ < 0.25 between peak X-ray flux and inter-event interval).

For every C/M/X flare we attach the GOES 1–8 Å flux nearest in time
(within ±5 min) to its catalogued peak time, then scatter peak flux against
the peak-to-peak recurrence time, coloured by active region and sized by
GOES class (C/M/X).

**Inputs**
- `data/goes_xray_flux_may2024.csv` — NOAA SWPC 1-minute GOES Longwave 1–8 Å
  series (`;`-separated, `,` decimal).
- `data/may_05_2-24_flares(complete)(with_C)_ars.xlsx` — flare catalogue.
  Columns used (0-indexed): 1 = peak time, 3 = GOES class, 4 = NOAA AR number,
  7 = recurrence (interval) time.

**Intermediate product**
- `data/flares_with_flux.csv` — one row per flare with mapped flux and
  recurrence time (regenerated by this notebook).

**Output**
- `figures/figure7_flux_vs_recurrence.png`


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path('..').resolve() / 'data'
FIG_DIR  = Path('..').resolve() / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

GOES_CSV    = DATA_DIR / 'goes_xray_flux_may2024.csv'
FLARES_XLSX = DATA_DIR / 'may_05_2-24_flares(complete)(with_C)_ars.xlsx'


In [ ]:
# --- Load GOES 1-8 A 1-minute series (';' separator, ',' decimal) ---
print('Loading GOES...')
goes = pd.read_csv(GOES_CSV, sep=';', decimal=',',
                   names=['timestamp', 'flux'], skiprows=1)
goes['timestamp'] = pd.to_datetime(goes['timestamp'], format='%Y-%m-%d %H:%M:%S')
goes = goes.sort_values('timestamp').reset_index(drop=True)
print(f'  {len(goes):,} GOES records')

# --- Load flare catalogue ---
print('Loading flares...')
flares = pd.read_excel(FLARES_XLSX)
print(f'  {len(flares)} flares')


def find_nearest_flux(peak_dt, goes_df, tol=5):
    """GOES flux closest in time to a flare peak (within +/- tol minutes)."""
    if pd.isna(peak_dt):
        return np.nan
    time_diffs = np.abs((goes_df['timestamp'] - peak_dt).dt.total_seconds() / 60)
    min_idx = time_diffs.idxmin()
    if time_diffs[min_idx] <= tol:
        return goes_df.loc[min_idx, 'flux']
    return np.nan


# Column positions in the workbook (0-indexed):
#   1 = peak time, 3 = GOES class, 4 = AR number, 7 = recurrence (interval) time
flares['Peak_Time_Converted'] = pd.to_datetime(flares.iloc[:, 1], errors='coerce')
flares['goes_flux'] = flares['Peak_Time_Converted'].apply(
    lambda x: find_nearest_flux(x, goes, tol=5))
flares['rt_hours'] = pd.to_timedelta(flares.iloc[:, 7],
                                     errors='coerce').dt.total_seconds() / 3600

output = pd.DataFrame({
    'Peak_Time':         flares['Peak_Time_Converted'],
    'GOES_Class':        flares.iloc[:, 3],
    'AR_Number':         flares.iloc[:, 4],
    'Recurrence_Time_h': flares['rt_hours'],
    'X_ray_Flux_Wm2':    flares['goes_flux'],
})
output.to_csv(DATA_DIR / 'flares_with_flux.csv', index=False)
print('Saved:', DATA_DIR / 'flares_with_flux.csv')

for ar in [13663, 13664]:
    a = output[output['AR_Number'] == ar]
    print(f'AR {ar}: {len(a)} flares | '
          f'mean RT = {a["Recurrence_Time_h"].mean():.2f} h | '
          f'mean flux = {a["X_ray_Flux_Wm2"].mean():.2e} W/m2')


In [ ]:
df = pd.read_csv(DATA_DIR / 'flares_with_flux.csv')
df = df.dropna(subset=['X_ray_Flux_Wm2', 'Recurrence_Time_h'])

# Drop temporal outliers with recurrence time > 10 h (paper convention)
df = df[df['Recurrence_Time_h'] < 10]

fig, ax = plt.subplots(figsize=(11, 8))

for ar, color, label in [(13663, '#1f77b4', 'AR 13663'),
                         (13664, '#d62728', 'AR 13664')]:
    ar_data = df[df['AR_Number'] == ar]

    # Marker size by GOES class: X largest, then M, then C
    sizes = [200 if str(c).startswith('X')
             else 120 if str(c).startswith('M')
             else 60
             for c in ar_data['GOES_Class']]

    ax.scatter(ar_data['Recurrence_Time_h'], ar_data['X_ray_Flux_Wm2'],
               s=sizes, alpha=0.7, color=color,
               edgecolors='black', linewidth=0.8,
               label=f'{label} (n={len(ar_data)})')

ax.set_xlabel('Flare Recurrence Time (hours)', fontsize=13, fontweight='bold')
ax.set_ylabel('X-ray Flux (W m$^{-2}$)', fontsize=13, fontweight='bold')
ax.set_yscale('log')
ax.set_xlim(0, 10)
ax.set_ylim(1e-6, 1e-3)
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(loc='upper right', fontsize=12, framealpha=0.95)

plt.tight_layout()
plt.savefig(FIG_DIR / 'figure7_flux_vs_recurrence.png', dpi=300, bbox_inches='tight')
plt.show()
